In [ ]:
# ========================================================
# 07_family_classification_analysis.ipynb
# Anomaly-family classification using Autoencoder embeddings
# ========================================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import matplotlib.pyplot as plt

print("=== KLASYFIKACJA SCENARIUSZY ANOMALNYCH NA EMBEDDINGACH AUTOENCODERA ===")

# ========================================================
# 1. Wczytanie schematu cech
# ========================================================

normal_df = pd.read_csv("../data/processed/normal_features.csv")
feature_columns = normal_df.columns.tolist()
input_dim = len(feature_columns)

print(f"Liczba cech wejściowych: {input_dim}")

# ========================================================
# 2. Definicja modelu Autoencoder
#    Musi być identyczna jak w notebooku 02
# ========================================================

class ImprovedAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
        )

        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# ========================================================
# 3. Wczytanie wytrenowanego modelu
# ========================================================

model_path = "../models/best_autoencoder_final.pth"

model = ImprovedAutoencoder(input_dim)
model.load_state_dict(torch.load(model_path, map_location="cpu"))
model.eval()

print(f"Model wczytany z: {model_path}")

# ========================================================
# 4. Funkcja do wyciągania embeddingów
# ========================================================

def get_embeddings(model, df):
    """
    Zwraca reprezentacje ukryte z warstwy bottleneck autoenkodera.

    WAŻNE:
    Dane w data/processed są już przeskalowane w notebooku 01.
    Dlatego tutaj NIE używamy StandardScaler i NIE robimy scaler.transform().
    """

    df = df.reindex(columns=feature_columns, fill_value=0)
    X = torch.tensor(df.values, dtype=torch.float32)

    model.eval()

    with torch.no_grad():
        embeddings = model.encoder(X)

    return embeddings.numpy()

# ========================================================
# 5. Przygotowanie zbioru do klasyfikacji
# ========================================================

scenarios = [
    "guloader",
    "scanning",
    "njrat",
    "kongtuke1",
    "kongtuke2",
    "remcos",
    "xloader",
    "xworm",
    "phantomstealer"
]

X_list = []
y_list = []

# Limit próbek na klasę.
# Dzięki temu scanning nie dominuje całego zbioru, bo ma dużo więcej flowów niż reszta.
MAX_PER_CLASS = 1000

print("\nTworzenie embeddingów:")

for scenario in scenarios:
    path = f"../data/processed/{scenario}_features.csv"
    df = pd.read_csv(path)
    df = df.reindex(columns=feature_columns, fill_value=0)

    if len(df) > MAX_PER_CLASS:
        df = df.sample(n=MAX_PER_CLASS, random_state=42)

    embeddings = get_embeddings(model, df)

    X_list.append(embeddings)
    y_list.extend([scenario] * len(embeddings))

    print(f"{scenario:15s} | liczba próbek: {len(embeddings)}")

X = np.vstack(X_list)
y = np.array(y_list)

print(f"\nŁączna liczba próbek: {X.shape[0]}")
print(f"Wymiar embeddingu: {X.shape[1]}")

# ========================================================
# 6. Podział na zbiór treningowy i testowy
# ========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print(f"\nZbiór treningowy: {X_train.shape[0]} próbek")
print(f"Zbiór testowy:    {X_test.shape[0]} próbek")

# ========================================================
# 7. Klasyfikator Random Forest
# ========================================================

clf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\n=== WYNIKI KLASYFIKACJI SCENARIUSZY ===")
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")

report_text = classification_report(y_test, y_pred)
print("\nClassification report:")
print(report_text)

# ========================================================
# 8. Macierz pomyłek
# ========================================================

cm = confusion_matrix(y_test, y_pred, labels=scenarios)

plt.figure(figsize=(10, 8))
plt.imshow(cm)
plt.title("Macierz pomyłek - klasyfikacja scenariuszy anomalnych")
plt.colorbar()

plt.xticks(range(len(scenarios)), scenarios, rotation=45, ha="right")
plt.yticks(range(len(scenarios)), scenarios)

plt.xlabel("Przewidziana klasa")
plt.ylabel("Rzeczywista klasa")

for i in range(len(scenarios)):
    for j in range(len(scenarios)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

# ========================================================
# 9. Zapis wyników do plików
# ========================================================

os.makedirs("../results", exist_ok=True)

report_dict = classification_report(y_test, y_pred, output_dict=True)
df_report = pd.DataFrame(report_dict).transpose()
df_report.to_csv("../results/family_classification_report.csv")

df_cm = pd.DataFrame(cm, index=scenarios, columns=scenarios)
df_cm.to_csv("../results/family_classification_confusion_matrix.csv")

summary = pd.DataFrame([{
    "accuracy": accuracy,
    "samples_total": len(y),
    "samples_train": len(y_train),
    "samples_test": len(y_test),
    "embedding_dim": X.shape[1],
    "max_per_class": MAX_PER_CLASS
}])

summary.to_csv("../results/family_classification_summary.csv", index=False)

print("\nZapisano:")
print("../results/family_classification_report.csv")
print("../results/family_classification_confusion_matrix.csv")
print("../results/family_classification_summary.csv")